# Season outputs loader
Helper cells to flexibly load season run outputs stored under `data/season_outputs/<run_id>`.
- Lists available runs
- Loads trip log, day summary, season person snapshots, SP day summary
- Discovers all `day_*_model_ts.parquet` files into a dict keyed by day index

Update `RUN_ID` below to point at the run you want to analyze.

In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)



In [ ]:
import importlib
import season.analysis_helpers as ah
import pandas as pd
from traffic.utils import animation_utils as au
import json, geopandas as gpd
from pprint import pprint
from notebooks.formatting import register_dollar_cols,register_elapsed_time

register_dollar_cols(tokens=["_toll_", "_rev", "_cost_", "_cost"])
register_elapsed_time(cols=['avg_tt', 'avg_tt_bus', 'avg_tt_car', 'avg_cum_time_lost', 'avg_cum_time_lost_bus', 'avg_cum_time_lost_car','avg_wait_bus', 'avg_onboard_time_bus',])


pd.set_option('display.max_columns', None)
# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2


In [ ]:
print("Available runs:")
ah.list_runs()

## Load a run
Set `RUN_ID` to one of the `available_runs` above.

In [ ]:
RUN_ID = 'runaway_repro' 


print('Available Data - List indicates multiple dfs')
run_data = ah.load_run(RUN_ID)



## Tier1 Metrics


### Summaries

In [ ]:
season_summary = ah.load_run_data_item(run_data, 'season_summary', day_index=None)
day_summary = ah.load_run_data_item(run_data, 'day_summary', day_index=None)


### Model Time Series Data

In [ ]:
model_ts = ah.load_run_data_item(run_data, 'model_ts', day_index=None)

### Per Person Metrics

In [ ]:
trip_log = ah.load_run_data_item(run_data, 'trip_log', day_index=None)
season_person_log = ah.load_run_data_item(run_data, 'season_person_log', day_index=None)

## Tier 1 Plotting

In [ ]:
ah.plot_model_ts_interactive(model_ts, run_id=RUN_ID)


In [ ]:
# from season.analysis_helpers import plot_realized_cost_means_with_total
# plot_realized_cost_means_with_total(trip_log)
# from season.analysis_helpers import plot_realized_cost_boxplots
# plot_realized_cost_boxplots(trip_log)


# Tier 2 data analysis

In [ ]:
DAY_INDEX = 2
road_gdf = ah.load_run_data_item(run_data, 'road_gdf', day_index=None)
spatial_df = ah.load_run_data_item(run_data, 'spatial', day_index=2)


In [ ]:
au.animate_relative_distance(spatial_df,2771, 100)

In [ ]:
au.animate_traffic(spatial_df, road_gdf=road_gdf, step_skip=2, watch=2771, zoom=10, color_by='has_person')

In [ ]:
au.animate_smoothed_spatial_metric(spatial_df, road_gdf, step_skip=10, metric='has_person')


In [ ]:
spatial_df.loc[(spatial_df.Step>9000)&(spatial_df.has_person)]
spatial_df.iloc[206317].pos == spatial_df.iloc[206318].pos


#spatial_df.loc[spatial_df.AgentID>2770].AgentID.drop_duplicates()

In [ ]:
row = spatial_df.loc[(~spatial_df["has_person"]) & (spatial_df["AgentType"] == "CarAgent")].iloc[0]
filtered_df = spatial_df[
    (spatial_df["Step"] == row["Step"]) &
    (spatial_df["AgentID"].between(row["AgentID"] - 2, row["AgentID"] + 2))
]
display(filtered_df)


In [ ]:
day2 = trip_log.loc[(trip_log['day_index'] == 2)&(trip_log['onboard_time'] > 80)]
day2

In [ ]:
# 

# to do 
add a df with all the vehicle agents and their attributes to the saved data, this should always be saved

does tt still start when the vehicle crosses the 0 distance threashhold